# Shared multi-seed / MSE-vs-KL training job

One shared bundle covers every assigned job (2 extra seeds each for vanilla, attention (λ2=1.0 MSE), and boundary (λ2=1.0, λ3=0.2), plus one KL run) — see `multiseed_kickoff.md` for the assignment table and your specific config values.

Upload **only** `multiseed_train_colab.zip`, not the whole repo.

**Before running:** Runtime → Change runtime type → GPU (T4 or better). Upload the zip to `MyDrive/multiseed_train_colab.zip`. Edit the config cell below to your assigned job, then Run all.

## Config — edit this to your assigned job (see multiseed_kickoff.md)

In [ ]:
# --- EDIT THESE FOR YOUR ASSIGNED JOB ---
VARIANT = "att"        # "vanilla" or "att"
SEED = 43              # 43 or 44 (or 42 for the KL job)
LAMBDA2 = 1.0          # fixed at the sweep winner for every job
ATT_MODE = "mse"       # "mse" or "kl" -- only matters when VARIANT == "att"
LAMBDA3 = 0.0          # 0.0 normally; 0.2 for the boundary jobs
OUTPUT_TAG = "att_mse_seed43"  # unique per job -- becomes the Drive output folder name
# ------------------------------------------

## Step 0: Unzip + deps

In [ ]:
import sys, zipfile, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
ZIP_ON_DRIVE = Path("/content/drive/MyDrive/multiseed_train_colab.zip")
BUNDLE = Path("/content/multiseed_train")
OUTPUTS = Path(f"/content/drive/MyDrive/multiseed_outputs_{OUTPUT_TAG}")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not (BUNDLE / "paths.py").exists():
        if not ZIP_ON_DRIVE.is_file():
            raise FileNotFoundError(f"Upload zip to {ZIP_ON_DRIVE}")
        print("Unzipping", ZIP_ON_DRIVE)
        with zipfile.ZipFile(ZIP_ON_DRIVE, "r") as z:
            z.extractall(BUNDLE.parent)
        if not (BUNDLE / "paths.py").exists():
            cands = [p for p in BUNDLE.parent.iterdir() if p.is_dir() and (p / "paths.py").exists()]
            if not cands:
                raise FileNotFoundError("paths.py not found after unzip")
            if BUNDLE.exists():
                shutil.rmtree(BUNDLE)
            shutil.move(str(cands[0]), str(BUNDLE))
    HERE = BUNDLE
else:
    HERE = Path.cwd()
    OUTPUTS = HERE / f"outputs_{OUTPUT_TAG}"

sys.path.insert(0, str(HERE))
OUTPUTS.mkdir(parents=True, exist_ok=True)
print("HERE =", HERE)
print("OUTPUTS =", OUTPUTS)
print(f"Job: variant={VARIANT} seed={SEED} lambda2={LAMBDA2} att_mode={ATT_MODE} lambda3={LAMBDA3}")

In [ ]:
import subprocess, sys
pkgs = ["transformers", "accelerate", "thop", "tqdm", "opencv-python-headless"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("deps ready")

## Step 1: Device check

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available():
    print("WARNING: no GPU -- this job is ~2h on T4; CPU will not finish in reasonable time.")

## Step 2: Train (this job only — ~2h on T4)

In [ ]:
import paths
from paths import add_teammate_paths, apply_data_dirs

paths.set_output_dirs(OUTPUTS / "checkpoints", OUTPUTS / "results")
add_teammate_paths()
apply_data_dirs()
print("CKPT_DIR:", paths.CKPT_DIR)
print("RESULTS_DIR:", paths.RESULTS_DIR)
assert paths.DATA_IMG_DIR.is_dir() and paths.DATA_MASK_DIR.is_dir()

import train_full_scale as T
import torch

class Args:
    n_train, n_val, n_test = 3576, 766, 766
    epochs = 20
    batch_size = 16
    lr = 6e-5
    lambda2 = LAMBDA2
    sigma = 8.0
    att_mode = ATT_MODE
    lambda3 = LAMBDA3
    boundary_kernel = 3
    seed = SEED

args = Args()
torch.manual_seed(args.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(args.seed)

best = paths.CKPT_DIR / f"segformer_b0_{VARIANT}_best.pt"
if best.exists():
    print(f"SKIP: {best} already exists")
else:
    T.train_variant(VARIANT, args)

## Step 3: Evaluate + collect results

In [ ]:
import eval_full_scale as E

class EvalArgs:
    n_train, n_val, n_test = 3576, 766, 766
    seed = SEED

row = E.evaluate_variant(VARIANT, EvalArgs())
print(f"{row['model']}: dice={row['dice']} iou={row['iou']} aamo={row['aamo']}")

print("\nOutputs under", OUTPUTS)
for p in sorted(OUTPUTS.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(OUTPUTS))

## Step 4: Send back

Share the whole `MyDrive/multiseed_outputs_<OUTPUT_TAG>/` folder (or zip it) back to Dhinanjaya — checkpoints + results for this one job.